# Practical 7 — Data Wrangling and Aggregation

## Objective

Transform the verified event-level dataset into trustworthy analysis-ready views without losing event identity, chronology, provenance, or minority-client behaviour.

**Simple meaning:** The existing data contains one row for every log event. In this practical, we will organise those events into meaningful summaries such as activity per day, per hour, and per client.

## Important distinction

- **Wrangling:** Reshaping and combining data into useful tables.
- **Aggregation:** Summarising multiple events into counts, rates, and statistics.
- **EDA:** Interpreting and visualising those summaries; this belongs mainly to Practical 8.

Aggregation can hide important behaviour. Therefore, we will preserve:

1. Event-level records for traceability.
2. Client-level summaries so one high-volume attacker does not represent every client.
3. Time-based summaries for detecting bursts and behavioural drift.
4. Separate descriptive and model-input datasets to prevent target leakage.

All aggregate totals must reconcile with the original 2,062,361 records.

## 1. Input contract and provenance chain

Practical 7 uses four verified event-aligned artifacts:

| Artifact | Purpose |
|---|---|
| Cleaned data | Original and normalized log fields |
| Weak labels | Attack, benign, uncertain, confidence, and conflict |
| Primary features | Event-local and past-only behavioural features |
| Split weights | Temporal split and training eligibility |

**Simple meaning:** Each file describes the same events from a different perspective.

Before joining them, we verify their manifests and fingerprints. Matching row counts alone are insufficient because two files can contain the same number of differently ordered records.

We will later join using both `event_id` and `record_hash`.

> The files are large, so Practical 7 will process them in chunks instead of loading all four complete files into memory.

In [1]:
from pathlib import Path
import json
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Project root containing data/ and src/ was not found.")

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))

CLEANED_FILE = PROJECT_ROOT / "data" / "processed" / "cj_cleaned.csv"
LABEL_FILE = PROJECT_ROOT / "data" / "labels" / "cj_weak_labels.csv"
FEATURE_FILE = PROJECT_ROOT / "data" / "features" / "cj_primary_features.csv"
BALANCING_FILE = PROJECT_ROOT / "data" / "balancing" / "cj_split_weights.csv"

CLEANING_MANIFEST = PROJECT_ROOT / "data" / "processed" / "cj_cleaning_manifest.json"
LABEL_MANIFEST = PROJECT_ROOT / "data" / "labels" / "cj_weak_labels_manifest.json"
FEATURE_MANIFEST = PROJECT_ROOT / "data" / "features" / "cj_primary_features_manifest.json"
BALANCING_MANIFEST = PROJECT_ROOT / "data" / "balancing" / "cj_split_weights_manifest.json"

artifact_paths = {
    "cleaned": CLEANED_FILE,
    "labels": LABEL_FILE,
    "features": FEATURE_FILE,
    "balancing": BALANCING_FILE,
}

manifest_paths = {
    "cleaning": CLEANING_MANIFEST,
    "labels": LABEL_MANIFEST,
    "features": FEATURE_MANIFEST,
    "balancing": BALANCING_MANIFEST,
}

missing_paths = [
    path
    for path in [*artifact_paths.values(), *manifest_paths.values()]
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing required artifacts:\n"
        + "\n".join(str(path) for path in missing_paths)
    )

with CLEANING_MANIFEST.open(encoding="utf-8") as file:
    cleaning_manifest = json.load(file)

with LABEL_MANIFEST.open(encoding="utf-8") as file:
    label_manifest = json.load(file)

with FEATURE_MANIFEST.open(encoding="utf-8") as file:
    feature_manifest = json.load(file)

with BALANCING_MANIFEST.open(encoding="utf-8") as file:
    balancing_manifest = json.load(file)

print("Project root:", PROJECT_ROOT)
print("\nRequired artifacts:")

for name, path in artifact_paths.items():
    print(f"{name:10s} {path.relative_to(PROJECT_ROOT)} "
          f"({path.stat().st_size / (1024 ** 2):,.2f} MiB)")

Project root: c:\Users\diyas\Desktop\PDS-Log-IDS-Project

Required artifacts:
cleaned    data\processed\cj_cleaned.csv (386.73 MiB)
labels     data\labels\cj_weak_labels.csv (216.91 MiB)
features   data\features\cj_primary_features.csv (371.61 MiB)
balancing  data\balancing\cj_split_weights.csv (136.51 MiB)


In [2]:
EXPECTED_ROWS = 2_062_361

row_counts = {
    "cleaned": cleaning_manifest["output_rows"],
    "labels": label_manifest["output_rows"],
    "features": feature_manifest["output_rows"],
    "balancing": balancing_manifest["output_rows"],
}

assert set(row_counts.values()) == {EXPECTED_ROWS}

assert (
    cleaning_manifest["output_sha256"]
    == label_manifest["input_sha256"]
    == feature_manifest["input_sha256"]
    == balancing_manifest["cleaned_sha256"]
)

assert (
    label_manifest["output_sha256"]
    == feature_manifest["label_sha256"]
    == balancing_manifest["label_sha256"]
)

assert (
    feature_manifest["output_sha256"]
    == balancing_manifest["feature_sha256"]
)

print("Artifact availability: passed")
print("Manifest row-count agreement: passed")
print("Cleaned-data provenance chain: passed")
print("Label provenance chain: passed")
print("Feature provenance chain: passed")
print("Expected records:", EXPECTED_ROWS)
print("Split version:", balancing_manifest["split_version"])

Artifact availability: passed
Manifest row-count agreement: passed
Cleaned-data provenance chain: passed
Label provenance chain: passed
Feature provenance chain: passed
Expected records: 2062361
Split version: calendar-temporal-v1


## 2. Chunk-wise identity alignment

A **sidecar** is a separate file that adds information to the same original records without duplicating the complete dataset.

Each artifact must contain the same event in the same row position:

- `event_id` identifies the particular occurrence.
- `record_hash` identifies its original content.

Using both fields detects missing, reordered, duplicated, or incorrectly connected records.

**Simple meaning:** Before combining information from different files, we confirm that every row refers to exactly the same log event.

The validation is performed chunk by chunk to keep memory usage controlled.

In [3]:
from itertools import zip_longest

import numpy as np
import pandas as pd

CHUNK_SIZE = 100_000
IDENTITY_COLUMNS = ["event_id", "record_hash"]

readers = {
    name: pd.read_csv(
        path,
        usecols=IDENTITY_COLUMNS,
        chunksize=CHUNK_SIZE,
        dtype="string",
        keep_default_na=False,
    )
    for name, path in artifact_paths.items()
}

aligned_rows = 0
verified_chunks = 0

for chunk_number, chunks in enumerate(
    zip_longest(*readers.values(), fillvalue=None),
    start=1,
):
    chunk_map = dict(zip(readers.keys(), chunks))

    if any(chunk is None for chunk in chunk_map.values()):
        raise AssertionError(
            f"Artifact chunk counts differ at chunk {chunk_number}."
        )

    reference = chunk_map["cleaned"][IDENTITY_COLUMNS]

    for artifact_name, chunk in chunk_map.items():
        if len(chunk) != len(reference):
            raise AssertionError(
                f"Row-count mismatch in {artifact_name}, "
                f"chunk {chunk_number}."
            )

        identity_matches = np.array_equal(
            reference.to_numpy(),
            chunk[IDENTITY_COLUMNS].to_numpy(),
        )

        if not identity_matches:
            different_rows = np.flatnonzero(
                np.any(
                    reference.to_numpy()
                    != chunk[IDENTITY_COLUMNS].to_numpy(),
                    axis=1,
                )
            )

            first_difference = int(different_rows[0])

            raise AssertionError(
                f"Identity mismatch in {artifact_name}, "
                f"chunk {chunk_number}, "
                f"local row {first_difference}."
            )

    aligned_rows += len(reference)
    verified_chunks += 1

    if chunk_number % 5 == 0:
        print(
            f"Verified {chunk_number} chunks "
            f"({aligned_rows:,} records)"
        )

assert aligned_rows == EXPECTED_ROWS

print("\nActual event identity alignment: passed")
print(f"Verified chunks: {verified_chunks}")
print(f"Verified records: {aligned_rows:,}")

Verified 5 chunks (500,000 records)
Verified 10 chunks (1,000,000 records)
Verified 15 chunks (1,500,000 records)
Verified 20 chunks (2,000,000 records)

Actual event identity alignment: passed
Verified chunks: 21
Verified records: 2,062,361


## 3. Aggregation contract

**Grain** means what exactly one row represents.

Defining the grain first prevents accidental double counting and ambiguous tables.

| Output view | One row represents | Main purpose |
|---|---|---|
| `hourly_summary` | One clock hour within one data split | Detect volume changes, bursts, and label drift |
| `client_split_summary` | One client within train, validation, or test | Compare client behaviour without mixing time periods |
| `client_daily_summary` | One client on one calendar day | Study short-term behavioural changes |
| `daily_summary` | One calendar day within one split | Produce understandable daily trends |

### Privacy decision

Raw client IP addresses must not appear in aggregate outputs. We will generate a keyed pseudonymous `client_key`.

A normal hash of an IP is insufficient because the limited IPv4 address space can be guessed. A keyed HMAC requires a private secret and is safer.

### Separation rule

These aggregates are initially **descriptive artifacts**, not automatic model features.

**Simple meaning:** They help us understand the data, but we will not feed them into a classifier until we prove that they contain no future information or label leakage.

In [4]:
AGGREGATION_CONTRACT = {
    "hourly_summary": {
        "grain": ["hour_start", "data_split"],
        "contains_client_key": False,
        "descriptive_only": True,
    },
    "daily_summary": {
        "grain": ["day", "data_split"],
        "contains_client_key": False,
        "descriptive_only": True,
    },
    "client_split_summary": {
        "grain": ["client_key", "data_split"],
        "contains_client_key": True,
        "descriptive_only": True,
    },
    "client_daily_summary": {
        "grain": ["client_key", "day", "data_split"],
        "contains_client_key": True,
        "descriptive_only": True,
    },
}

contract_rows = [
    {
        "output_view": view_name,
        "grain": " + ".join(properties["grain"]),
        "contains_client_key": properties["contains_client_key"],
        "descriptive_only": properties["descriptive_only"],
    }
    for view_name, properties in AGGREGATION_CONTRACT.items()
]

contract_table = pd.DataFrame(contract_rows)

assert contract_table["output_view"].is_unique
assert contract_table["descriptive_only"].all()

contract_table

,output_view,grain,contains_client_key,descriptive_only
0,hourly_summary,hour_start + data_split,False,True
1,daily_summary,day + data_split,False,True
2,client_split_summary,client_key + data_split,True,True
3,client_daily_summary,client_key + day + data_split,True,True


## 4. Privacy-safe client identity

Client behaviour must remain linkable across events, but raw IP addresses must not appear in aggregate outputs.

We use **HMAC-SHA256**:

- The same IP and secret produce the same `client_key`.
- Different IPs produce different keys.
- The original IP cannot practically be recovered without the secret.
- The secret remains only in the local `.env` file.

HMAC is pseudonymisation, not anonymisation. Anyone possessing both the secret and candidate IP addresses can reproduce the keys.

In [6]:
import hashlib
import hmac
import os

ENV_FILE = PROJECT_ROOT / ".env"

def load_local_secret(variable_name: str) -> str:
    environment_value = os.environ.get(variable_name)
    if environment_value:
        return environment_value

    if not ENV_FILE.exists():
        raise FileNotFoundError("Local .env file was not found.")

    for line in ENV_FILE.read_text(encoding="utf-8-sig").splitlines():
        stripped = line.strip()

        if not stripped or stripped.startswith("#") or "=" not in stripped:
            continue

        name, value = stripped.split("=", maxsplit=1)

        if name.strip() == variable_name:
            return value.strip()

    raise KeyError(f"{variable_name} was not found in the environment or .env file.")

CLIENT_HMAC_KEY = load_local_secret("CLIENT_HMAC_KEY")

if len(CLIENT_HMAC_KEY) < 32:
    raise ValueError("CLIENT_HMAC_KEY is unexpectedly short.")

def create_client_key(client_ip: str) -> str:
    normalized_ip = str(client_ip).strip()

    if not normalized_ip:
        raise ValueError("Client IP cannot be empty.")

    digest = hmac.new(
        CLIENT_HMAC_KEY.encode("utf-8"),
        normalized_ip.encode("utf-8"),
        hashlib.sha256,
    ).hexdigest()

    return f"client_{digest[:24]}"

In [7]:
first_key = create_client_key("192.0.2.1")
repeated_key = create_client_key("192.0.2.1")
different_key = create_client_key("198.51.100.2")

assert first_key == repeated_key
assert first_key != different_key
assert first_key.startswith("client_")
assert len(first_key) == 31
assert "192.0.2.1" not in first_key

print("Deterministic pseudonymisation: passed")
print("Different-client separation: passed")
print("Raw-IP exclusion: passed")
print("Secret loaded without displaying it: passed")

Deterministic pseudonymisation: passed
Different-client separation: passed
Raw-IP exclusion: passed
Secret loaded without displaying it: passed


## 5. Synthetic contract test

Before processing the real dataset, we test the wrangling function using artificial records.

This verifies:

- aligned sidecars are accepted;
- misaligned identities are rejected;
- repeated clients receive the same pseudonym;
- raw IP fields do not enter the prepared output;
- timestamps are converted into hourly and daily grains.

**Simple meaning:** A small controlled test catches structural mistakes quickly and without exposing real data.

In [10]:
import importlib
import src.aggregation as aggregation

importlib.reload(aggregation)

synthetic_cleaned = pd.DataFrame({
    "event_id": ["event-1", "event-2", "event-3"],
    "record_hash": ["hash-a", "hash-b", "hash-c"],
    "timestamp_normalized": [
        "2024-01-01 10:05:00",
        "2024-01-01 10:45:00",
        "2024-01-02 09:00:00",
    ],
    "client_ip_canonical": [
        "192.0.2.1",
        "192.0.2.1",
        "198.51.100.2",
    ],
    "source_port_normalized": [1200, 1201, 53000],
})

synthetic_labels = pd.DataFrame({
    "event_id": ["event-1", "event-2", "event-3"],
    "record_hash": ["hash-a", "hash-b", "hash-c"],
    "weak_label": ["benign", "attack", "uncertain"],
    "label_confidence": [0.55, 0.85, 0.00],
    "label_conflict": [False, True, False],
})

synthetic_features = pd.DataFrame({
    "event_id": ["event-1", "event-2", "event-3"],
    "record_hash": ["hash-a", "hash-b", "hash-c"],
    "timestamp_reversal_flag": [0, 0, 1],
    "activity_decay_60s": [0.0, 1.2, 0.0],
    "activity_decay_300s": [0.0, 1.8, 0.0],
    "activity_decay_3600s": [0.0, 2.1, 0.0],
})

synthetic_balancing = pd.DataFrame({
    "event_id": ["event-1", "event-2", "event-3"],
    "record_hash": ["hash-a", "hash-b", "hash-c"],
    "data_split": ["train", "train", "validation"],
})

synthetic_cache = {}

synthetic_result = aggregation.prepare_analysis_chunk(
    cleaned=synthetic_cleaned,
    labels=synthetic_labels,
    features=synthetic_features,
    balancing=synthetic_balancing,
    client_hmac_key=CLIENT_HMAC_KEY,
    client_key_cache=synthetic_cache,
)

assert len(synthetic_result) == 3
assert synthetic_result.loc[0, "client_key"] == synthetic_result.loc[1, "client_key"]
assert synthetic_result.loc[0, "client_key"] != synthetic_result.loc[2, "client_key"]
assert synthetic_result.loc[0, "hour_start"] == pd.Timestamp("2024-01-01 10:00:00")
assert synthetic_result.loc[0, "day"] == pd.Timestamp("2024-01-01")
assert synthetic_result["label_conflict"].tolist() == [0, 1, 0]

for forbidden_name in [
    "client_ip",
    "client_ip_canonical",
    "user_agent",
    "metadata",
]:
    assert forbidden_name not in synthetic_result.columns

misaligned_labels = synthetic_labels.copy()
misaligned_labels.loc[1, "event_id"] = "wrong-event"

try:
    aggregation.prepare_analysis_chunk(
        cleaned=synthetic_cleaned,
        labels=misaligned_labels,
        features=synthetic_features,
        balancing=synthetic_balancing,
        client_hmac_key=CLIENT_HMAC_KEY,
        client_key_cache={},
    )
except ValueError as error:
    assert "identity mismatch" in str(error)
else:
    raise AssertionError("Misaligned identity was not rejected.")

print("Aligned preparation: passed")
print("Misalignment rejection: passed")
print("Pseudonym consistency: passed")
print("Sensitive-field exclusion: passed")
print("Time-grain derivation: passed")

synthetic_result

Aligned preparation: passed
Misalignment rejection: passed
Pseudonym consistency: passed
Sensitive-field exclusion: passed
Time-grain derivation: passed


,event_id,record_hash,timestamp,hour_start,day,client_key,source_port,weak_label,label_confidence,label_conflict,timestamp_reversal_flag,activity_decay_60s,activity_decay_300s,activity_decay_3600s,data_split
0,event-1,hash-a,2024-01-01 10:05:00,2024-01-01 10:00:00,2024-01-01,client_6be41037f4e52f4e9953034c,1200,benign,0.55,0,0,0.0,0.0,0.0,train
1,event-2,hash-b,2024-01-01 10:45:00,2024-01-01 10:00:00,2024-01-01,client_6be41037f4e52f4e9953034c,1201,attack,0.85,1,0,1.2,1.8,2.1,train
2,event-3,hash-c,2024-01-02 09:00:00,2024-01-02 09:00:00,2024-01-02,client_eb3b63cdcd0477a1bc60087c,53000,uncertain,0.00,0,1,0.0,0.0,0.0,validation


## 6. Chunk-invariant time aggregation

Chunking is only a memory-management technique; changing the chunk boundary must not change the result.

Counts can be added across chunks, but unique-client counts cannot be added directly because the same client may occur on both sides of a chunk boundary. We therefore maintain a set of client identities for every time grain.

This property is called **chunk-boundary invariance**.

In [12]:
importlib.reload(aggregation)

single_hour_state = aggregation.create_time_aggregation_state()
aggregation.update_time_aggregation_state(
    single_hour_state,
    synthetic_result,
    "hour_start",
)
single_hour_result = aggregation.finalize_time_aggregation(
    single_hour_state,
    "hour_start",
)

split_hour_state = aggregation.create_time_aggregation_state()
aggregation.update_time_aggregation_state(
    split_hour_state,
    synthetic_result.iloc[:1],
    "hour_start",
)
aggregation.update_time_aggregation_state(
    split_hour_state,
    synthetic_result.iloc[1:],
    "hour_start",
)
split_hour_result = aggregation.finalize_time_aggregation(
    split_hour_state,
    "hour_start",
)

pd.testing.assert_frame_equal(
    single_hour_result,
    split_hour_result,
)

first_hour = split_hour_result.iloc[0]

assert int(split_hour_result["event_count"].sum()) == 3
assert first_hour["event_count"] == 2
assert first_hour["unique_client_count"] == 1
assert first_hour["attack_count"] == 1
assert first_hour["benign_count"] == 1
assert first_hour["conflict_count"] == 1

print("Event-count reconciliation: passed")
print("Label-count reconciliation: passed")
print("Exact unique-client counting: passed")
print("Chunk-boundary invariance: passed")

split_hour_result

Event-count reconciliation: passed
Label-count reconciliation: passed
Exact unique-client counting: passed
Chunk-boundary invariance: passed


,hour_start,data_split,event_count,unique_client_count,attack_count,benign_count,uncertain_count,attack_percent,benign_percent,uncertain_percent,conflict_count,conflict_percent,mean_label_confidence,timestamp_reversal_count
0,2024-01-01 10:00:00,train,2,1,1,1,0,50.0,50.0,0.0,1,50.0,0.7,0
1,2024-01-02 09:00:00,validation,1,1,0,0,1,0.0,0.0,100.0,0,0.0,0.0,1


## 7. Client-centred summaries

Event-weighted analysis asks: **What proportion of all log events has a property?**

Client-weighted analysis asks: **What proportion of clients exhibits that property?**

Both views are necessary. One attacker generated hundreds of thousands of events, so an event-weighted result can be dominated by that single client. Client summaries allow every client or client-period to become one analysis unit.

The client aggregates also retain active duration, unique-port count, conflicts, label composition, and past-only activity intensity.

In [13]:
importlib.reload(aggregation)

single_client_state = aggregation.create_client_aggregation_state()
aggregation.update_client_aggregation_state(
    single_client_state,
    synthetic_result,
    ["client_key", "data_split"],
)
single_client_result = aggregation.finalize_client_aggregation(
    single_client_state,
    ["client_key", "data_split"],
)

split_client_state = aggregation.create_client_aggregation_state()
aggregation.update_client_aggregation_state(
    split_client_state,
    synthetic_result.iloc[:1],
    ["client_key", "data_split"],
)
aggregation.update_client_aggregation_state(
    split_client_state,
    synthetic_result.iloc[1:],
    ["client_key", "data_split"],
)
split_client_result = aggregation.finalize_client_aggregation(
    split_client_state,
    ["client_key", "data_split"],
)

pd.testing.assert_frame_equal(
    single_client_result,
    split_client_result,
)

train_client = split_client_result[
    split_client_result["data_split"] == "train"
].iloc[0]

assert int(split_client_result["event_count"].sum()) == 3
assert train_client["event_count"] == 2
assert train_client["unique_port_count"] == 2
assert train_client["attack_count"] == 1
assert train_client["benign_count"] == 1
assert train_client["active_span_seconds"] == 2400.0

for forbidden_name in [
    "client_ip",
    "client_ip_canonical",
]:
    assert forbidden_name not in split_client_result.columns

print("Client event reconciliation: passed")
print("Unique-port calculation: passed")
print("Active-span calculation: passed")
print("Chunk-boundary invariance: passed")
print("Raw-IP exclusion: passed")

split_client_result

Client event reconciliation: passed
Unique-port calculation: passed
Active-span calculation: passed
Chunk-boundary invariance: passed
Raw-IP exclusion: passed


,client_key,data_split,event_count,first_timestamp,last_timestamp,active_span_seconds,unique_port_count,attack_count,benign_count,uncertain_count,attack_percent,benign_percent,uncertain_percent,conflict_count,conflict_percent,mean_label_confidence,timestamp_reversal_count,mean_activity_decay_60s,mean_activity_decay_300s,mean_activity_decay_3600s
0,client_6be41037f4e52f4e9953034c,train,2,2024-01-01 10:05:00,2024-01-01 10:45:00,2400.0,2,1,1,0,50.0,50.0,0.0,1,50.0,0.7,0,0.6,0.9,1.05
1,client_eb3b63cdcd0477a1bc60087c,validation,1,2024-01-02 09:00:00,2024-01-02 09:00:00,0.0,1,0,0,1,0.0,0.0,100.0,0,0.0,0.0,1,0.0,0.0,0.00


## 8. Real-data preflight

The first 100,000 consecutive events are used as a structural preflight.

This is not a representative statistical sample. Its purpose is to verify schema compatibility, privacy, alignment, aggregation grains, and count reconciliation using real records.

A contiguous section is appropriate here because we are testing a temporal process. Random sampling would break event continuity.

In [14]:
importlib.reload(aggregation)

PREFLIGHT_ROWS = 100_000

preflight_cleaned = pd.read_csv(
    CLEANED_FILE,
    usecols=aggregation.CLEANED_COLUMNS,
    nrows=PREFLIGHT_ROWS,
    keep_default_na=False,
)

preflight_labels = pd.read_csv(
    LABEL_FILE,
    usecols=aggregation.LABEL_COLUMNS,
    nrows=PREFLIGHT_ROWS,
    keep_default_na=False,
)

preflight_features = pd.read_csv(
    FEATURE_FILE,
    usecols=aggregation.FEATURE_COLUMNS,
    nrows=PREFLIGHT_ROWS,
    keep_default_na=False,
)

preflight_balancing = pd.read_csv(
    BALANCING_FILE,
    usecols=aggregation.BALANCING_COLUMNS,
    nrows=PREFLIGHT_ROWS,
    keep_default_na=False,
)

preflight_client_cache = {}

preflight_analysis = aggregation.prepare_analysis_chunk(
    cleaned=preflight_cleaned,
    labels=preflight_labels,
    features=preflight_features,
    balancing=preflight_balancing,
    client_hmac_key=CLIENT_HMAC_KEY,
    client_key_cache=preflight_client_cache,
)

hourly_state = aggregation.create_time_aggregation_state()
daily_state = aggregation.create_time_aggregation_state()
client_split_state = aggregation.create_client_aggregation_state()
client_daily_state = aggregation.create_client_aggregation_state()

aggregation.update_time_aggregation_state(
    hourly_state,
    preflight_analysis,
    "hour_start",
)

aggregation.update_time_aggregation_state(
    daily_state,
    preflight_analysis,
    "day",
)

aggregation.update_client_aggregation_state(
    client_split_state,
    preflight_analysis,
    ["client_key", "data_split"],
)

aggregation.update_client_aggregation_state(
    client_daily_state,
    preflight_analysis,
    ["client_key", "day", "data_split"],
)

preflight_hourly = aggregation.finalize_time_aggregation(
    hourly_state,
    "hour_start",
)

preflight_daily = aggregation.finalize_time_aggregation(
    daily_state,
    "day",
)

preflight_client_split = aggregation.finalize_client_aggregation(
    client_split_state,
    ["client_key", "data_split"],
)

preflight_client_daily = aggregation.finalize_client_aggregation(
    client_daily_state,
    ["client_key", "day", "data_split"],
)

In [15]:
preflight_outputs = {
    "hourly": preflight_hourly,
    "daily": preflight_daily,
    "client_split": preflight_client_split,
    "client_daily": preflight_client_daily,
}

assert len(preflight_analysis) == PREFLIGHT_ROWS
assert preflight_analysis["event_id"].is_unique

for output_name, output_frame in preflight_outputs.items():
    reconciled_events = int(output_frame["event_count"].sum())

    assert reconciled_events == PREFLIGHT_ROWS, (
        f"{output_name} reconciled {reconciled_events:,} "
        f"instead of {PREFLIGHT_ROWS:,}"
    )

assert not preflight_hourly.duplicated(
    ["hour_start", "data_split"]
).any()

assert not preflight_daily.duplicated(
    ["day", "data_split"]
).any()

assert not preflight_client_split.duplicated(
    ["client_key", "data_split"]
).any()

assert not preflight_client_daily.duplicated(
    ["client_key", "day", "data_split"]
).any()

for forbidden_column in [
    "client_ip",
    "client_ip_canonical",
    "user_agent",
    "metadata",
]:
    assert forbidden_column not in preflight_analysis.columns

assert len(preflight_client_cache) == (
    preflight_analysis["client_key"].nunique()
)

print(f"Prepared records: {len(preflight_analysis):,}")
print(f"Pseudonymous clients: {len(preflight_client_cache):,}")
print(f"Hourly rows: {len(preflight_hourly):,}")
print(f"Daily rows: {len(preflight_daily):,}")
print(f"Client-split rows: {len(preflight_client_split):,}")
print(f"Client-day rows: {len(preflight_client_daily):,}")
print("Four-view event reconciliation: passed")
print("Grain uniqueness: passed")
print("Sensitive-field exclusion: passed")

preflight_daily.head()

Prepared records: 100,000
Pseudonymous clients: 7,997
Hourly rows: 3,940
Daily rows: 165
Client-split rows: 7,997
Client-day rows: 14,458
Four-view event reconciliation: passed
Grain uniqueness: passed
Sensitive-field exclusion: passed


,day,data_split,event_count,unique_client_count,attack_count,benign_count,uncertain_count,attack_percent,benign_percent,uncertain_percent,conflict_count,conflict_percent,mean_label_confidence,timestamp_reversal_count
0,2023-01-08,train,251,95,11,194,46,4.382470,77.290837,18.326693,8,3.187251,0.463147,0
1,2023-01-09,train,3018,112,1948,888,182,64.546057,29.423459,6.030484,1931,63.982770,0.719211,0
2,2023-01-10,train,5705,101,5228,288,189,91.638913,5.048203,3.312883,5210,91.323401,0.814272,0
3,2023-01-11,train,5801,105,2467,2991,343,42.527150,51.560076,5.912774,2467,42.527150,0.646051,0
4,2023-01-12,train,246,85,34,142,70,13.821138,57.723577,28.455285,33,13.414634,0.435366,0


### Preflight conclusion

The real-data preflight passed:

- 100,000 consecutive events were processed.
- 7,997 pseudonymous clients were preserved.
- Four different aggregation grains reconciled to the same event total.
- No duplicate grain keys were created.
- Raw client IP fields were excluded.

The 14,458 client-day rows show that one client may contribute behaviour on multiple days. These rows must not be interpreted as 14,458 different clients.

The preflight validates implementation correctness; it does not estimate the final dataset distribution.

## 9. Atomic and idempotent export

An **atomic export** writes temporary files first and publishes them only after successful completion. This prevents incomplete output from appearing valid.

An **idempotent export** safely reuses an already verified result when the inputs, code contract, and pseudonym key remain unchanged.

The manifest stores only a short fingerprint of the pseudonym key—not the secret itself.

In [16]:
import tempfile

importlib.reload(aggregation)

AGGREGATED_DIRECTORY = (
    PROJECT_ROOT / "data" / "aggregated"
)

with tempfile.TemporaryDirectory(
    prefix="_synthetic_export_test_",
    dir=AGGREGATED_DIRECTORY,
) as temporary_directory:
    test_root = Path(temporary_directory)
    test_inputs = test_root / "inputs"
    test_outputs = test_root / "outputs"

    test_inputs.mkdir()
    test_outputs.mkdir()

    synthetic_paths = {
        "cleaned": test_inputs / "cleaned.csv",
        "labels": test_inputs / "labels.csv",
        "features": test_inputs / "features.csv",
        "balancing": test_inputs / "balancing.csv",
    }

    synthetic_cleaned.to_csv(
        synthetic_paths["cleaned"],
        index=False,
    )
    synthetic_labels.to_csv(
        synthetic_paths["labels"],
        index=False,
    )
    synthetic_features.to_csv(
        synthetic_paths["features"],
        index=False,
    )
    synthetic_balancing.to_csv(
        synthetic_paths["balancing"],
        index=False,
    )

    synthetic_fingerprints = {
        name: aggregation.calculate_file_sha256(path)
        for name, path in synthetic_paths.items()
    }

    first_export = aggregation.export_aggregated_views(
        project_root=PROJECT_ROOT,
        cleaned_file=synthetic_paths["cleaned"],
        label_file=synthetic_paths["labels"],
        feature_file=synthetic_paths["features"],
        balancing_file=synthetic_paths["balancing"],
        output_directory=test_outputs,
        client_hmac_key=CLIENT_HMAC_KEY,
        source_fingerprints=synthetic_fingerprints,
        expected_rows=3,
        chunk_size=1,
    )

    second_export = aggregation.export_aggregated_views(
        project_root=PROJECT_ROOT,
        cleaned_file=synthetic_paths["cleaned"],
        label_file=synthetic_paths["labels"],
        feature_file=synthetic_paths["features"],
        balancing_file=synthetic_paths["balancing"],
        output_directory=test_outputs,
        client_hmac_key=CLIENT_HMAC_KEY,
        source_fingerprints=synthetic_fingerprints,
        expected_rows=3,
        chunk_size=1,
    )

    assert first_export["export_status"] == "created"
    assert second_export["export_status"] == "reused"
    assert first_export["input_rows"] == 3
    assert first_export["chunk_count"] == 3
    assert first_export["raw_ip_exported"] is False
    assert first_export["descriptive_only"] is True
    assert first_export["unique_clients"] == 2
    assert "CLIENT_HMAC_KEY" not in str(first_export)
    assert CLIENT_HMAC_KEY not in str(first_export)

    for output_name, metadata in first_export["outputs"].items():
        output_path = PROJECT_ROOT / metadata["file"]
        exported_frame = pd.read_csv(output_path)

        assert (
            int(exported_frame["event_count"].sum()) == 3
        )

        assert "client_ip" not in exported_frame.columns
        assert (
            "client_ip_canonical"
            not in exported_frame.columns
        )

print("Atomic synthetic export: passed")
print("Manifest generation: passed")
print("Verified-output reuse: passed")
print("Secret exclusion: passed")
print("Synthetic temporary files removed: passed")

Atomic synthetic export: passed
Manifest generation: passed
Verified-output reuse: passed
Secret exclusion: passed
Synthetic temporary files removed: passed


## 10. Full one-pass aggregation

The four source artifacts are read together in aligned chunks. Each chunk updates all four aggregation views before the next chunk is loaded.

This is called **one-pass fan-out**:

- large source files are read only once;
- every output observes the same verified events;
- memory usage remains bounded;
- all views must reconcile to 2,062,361 events.

Generated files are published only after every chunk and reconciliation check succeeds.

In [17]:
importlib.reload(aggregation)

SOURCE_FINGERPRINTS = {
    "cleaned": cleaning_manifest["output_sha256"],
    "labels": label_manifest["output_sha256"],
    "features": feature_manifest["output_sha256"],
    "balancing": balancing_manifest["output_sha256"],
}

aggregation_result = aggregation.export_aggregated_views(
    project_root=PROJECT_ROOT,
    cleaned_file=CLEANED_FILE,
    label_file=LABEL_FILE,
    feature_file=FEATURE_FILE,
    balancing_file=BALANCING_FILE,
    output_directory=AGGREGATED_DIRECTORY,
    client_hmac_key=CLIENT_HMAC_KEY,
    source_fingerprints=SOURCE_FINGERPRINTS,
    expected_rows=EXPECTED_ROWS,
    chunk_size=CHUNK_SIZE,
)

Processed 5 chunks (500,000 events)
Processed 10 chunks (1,000,000 events)
Processed 15 chunks (1,500,000 events)
Processed 20 chunks (2,000,000 events)


In [18]:
print("Export status:", aggregation_result["export_status"])
print(f"Input events: {aggregation_result['input_rows']:,}")
print("Processed chunks:", aggregation_result["chunk_count"])
print(f"Unique clients: {aggregation_result['unique_clients']:,}")
print("Minimum timestamp:", aggregation_result["minimum_timestamp"])
print("Maximum timestamp:", aggregation_result["maximum_timestamp"])
print("Raw IP exported:", aggregation_result["raw_ip_exported"])
print("Descriptive only:", aggregation_result["descriptive_only"])
print("Elapsed seconds:", aggregation_result["elapsed_seconds"])

print("\nGenerated views:")

for view_name, metadata in aggregation_result["outputs"].items():
    print(
        f"{view_name:22s} "
        f"rows={metadata['rows']:,} "
        f"sha256={metadata['sha256'][:12]}..."
    )

Export status: created
Input events: 2,062,361
Processed chunks: 21
Unique clients: 16,680
Minimum timestamp: 2023-01-08T08:07:15
Maximum timestamp: 2024-02-19T21:44:01
Raw IP exported: False
Descriptive only: True
Elapsed seconds: 60.57

Generated views:
hourly_summary         rows=9,778 sha256=746fba59f424...
daily_summary          rows=408 sha256=48c5ad18b8a2...
client_split_summary   rows=17,607 sha256=17161742c863...
client_daily_summary   rows=32,809 sha256=0d699daed181...


## 11. Independent artifact verification

Verification must read the saved files from disk instead of trusting the in-memory objects that created them.

This catches:

- incomplete or altered files;
- duplicate aggregation grains;
- incorrect label totals;
- temporal split leakage;
- accidentally exported identifiers;
- mismatches between files and their manifest.

**Simple meaning:** We inspect the delivered result independently, like another person checking our work.

In [19]:
AGGREGATED_DIRECTORY = PROJECT_ROOT / "data" / "aggregated"
aggregation_output_paths = {
    "hourly_summary":
        AGGREGATED_DIRECTORY / "cj_hourly_summary.csv",
    "daily_summary":
        AGGREGATED_DIRECTORY / "cj_daily_summary.csv",
    "client_split_summary":
        AGGREGATED_DIRECTORY / "cj_client_split_summary.csv",
    "client_daily_summary":
        AGGREGATED_DIRECTORY / "cj_client_daily_summary.csv",
}

aggregation_manifest_path = (
    AGGREGATED_DIRECTORY / "cj_aggregation_manifest.json"
)

with aggregation_manifest_path.open(encoding="utf-8") as file:
    saved_aggregation_manifest = json.load(file)

saved_aggregates = {
    "hourly_summary": pd.read_csv(
        aggregation_output_paths["hourly_summary"],
        parse_dates=["hour_start"],
    ),
    "daily_summary": pd.read_csv(
        aggregation_output_paths["daily_summary"],
        parse_dates=["day"],
    ),
    "client_split_summary": pd.read_csv(
        aggregation_output_paths["client_split_summary"],
        parse_dates=["first_timestamp", "last_timestamp"],
    ),
    "client_daily_summary": pd.read_csv(
        aggregation_output_paths["client_daily_summary"],
        parse_dates=["day", "first_timestamp", "last_timestamp"],
    ),
}

saved_grains = {
    "hourly_summary": ["hour_start", "data_split"],
    "daily_summary": ["day", "data_split"],
    "client_split_summary": ["client_key", "data_split"],
    "client_daily_summary": [
        "client_key",
        "day",
        "data_split",
    ],
}

for view_name, frame in saved_aggregates.items():
    expected_sha256 = saved_aggregation_manifest[
        "outputs"
    ][view_name]["sha256"]

    actual_sha256 = aggregation.calculate_file_sha256(
        aggregation_output_paths[view_name]
    )

    assert actual_sha256 == expected_sha256
    assert int(frame["event_count"].sum()) == EXPECTED_ROWS
    assert not frame.duplicated(saved_grains[view_name]).any()

    label_total = (
        frame["attack_count"]
        + frame["benign_count"]
        + frame["uncertain_count"]
    )

    assert label_total.equals(frame["event_count"])

    label_percent_total = (
        frame["attack_percent"]
        + frame["benign_percent"]
        + frame["uncertain_percent"]
    )

    assert np.allclose(
        label_percent_total,
        100.0,
        atol=1e-8,
    )

    forbidden_columns = {
        "client_ip",
        "client_ip_canonical",
        "user_agent",
        "metadata",
    }

    assert not forbidden_columns.intersection(frame.columns)

for view_name in [
    "client_split_summary",
    "client_daily_summary",
]:
    client_keys = saved_aggregates[view_name]["client_key"]

    assert client_keys.str.fullmatch(
        r"client_[0-9a-f]{24}"
    ).all()

validation_start = pd.Timestamp(
    balancing_manifest["validation_start"]
)

test_start = pd.Timestamp(
    balancing_manifest["test_start"]
)

def reconstruct_split(timestamps: pd.Series) -> pd.Series:
    return pd.Series(
        np.select(
            [
                timestamps < validation_start,
                timestamps < test_start,
            ],
            [
                "train",
                "validation",
            ],
            default="test",
        ),
        index=timestamps.index,
    )

assert reconstruct_split(
    saved_aggregates["hourly_summary"]["hour_start"]
).equals(
    saved_aggregates["hourly_summary"]["data_split"]
)

assert reconstruct_split(
    saved_aggregates["daily_summary"]["day"]
).equals(
    saved_aggregates["daily_summary"]["data_split"]
)

assert reconstruct_split(
    saved_aggregates["client_daily_summary"]["day"]
).equals(
    saved_aggregates["client_daily_summary"]["data_split"]
)

client_split = saved_aggregates["client_split_summary"]

for split_name, group in client_split.groupby("data_split"):
    if split_name == "train":
        assert (group["last_timestamp"] < validation_start).all()
    elif split_name == "validation":
        assert (group["first_timestamp"] >= validation_start).all()
        assert (group["last_timestamp"] < test_start).all()
    elif split_name == "test":
        assert (group["first_timestamp"] >= test_start).all()
    else:
        raise AssertionError(f"Unexpected split: {split_name}")

assert (
    client_split["client_key"].nunique()
    == saved_aggregation_manifest["unique_clients"]
    == 16_680
)

manifest_text = aggregation_manifest_path.read_text(
    encoding="utf-8"
)

assert CLIENT_HMAC_KEY not in manifest_text

partial_files = list(
    AGGREGATED_DIRECTORY.glob("*.partial")
)

assert not partial_files

print(f"Verified events per view: {EXPECTED_ROWS:,}")
print("Output fingerprints: passed")
print("Grain uniqueness: passed")
print("Label reconciliation: passed")
print("Percentage reconciliation: passed")
print("Calendar-split reconstruction: passed")
print("Pseudonym format and cardinality: passed")
print("Raw-field and secret exclusion: passed")
print("Partial files remaining:", len(partial_files))

Verified events per view: 2,062,361
Output fingerprints: passed
Grain uniqueness: passed
Label reconciliation: passed
Percentage reconciliation: passed
Calendar-split reconstruction: passed
Pseudonym format and cardinality: passed
Raw-field and secret exclusion: passed
Partial files remaining: 0


## 12. Training-only behavioural-regime threshold

We want to distinguish:

- `ordinary` — normal active-hour volume;
- `burst` — unusually high event volume;
- `post_burst` — the immediately following active hour after a burst.

The burst threshold must be learned only from training hours. Using validation or test activity to choose it would leak future behaviour into the methodology.

The regime uses event volume, not weak labels. Therefore, it does not circularly define a burst as an attack merely because the weak label says attack.

An **active hour** is an hour containing at least one recorded event. Missing hours are not silently treated as zero-volume observations.

In [21]:
verified_hourly = saved_aggregates["hourly_summary"].copy()

training_hourly = verified_hourly[
    verified_hourly["data_split"] == "train"
].copy()

training_start = training_hourly["hour_start"].min()
training_end = training_hourly["hour_start"].max()

possible_training_hours = int(
    (training_end - training_start).total_seconds() / 3600
) + 1

active_training_hours = len(training_hourly)

inactive_training_hours = (
    possible_training_hours - active_training_hours
)

training_quantiles = (
    training_hourly["event_count"]
    .quantile([
        0.00,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.995,
        1.00,
    ])
    .rename("event_count")
    .to_frame()
)

print(f"Training start: {training_start}")
print(f"Training end: {training_end}")
print(f"Possible training hours: {possible_training_hours:,}")
print(f"Active training hours: {active_training_hours:,}")
print(f"Hours without records: {inactive_training_hours:,}")

training_quantiles

Training start: 2023-01-08 08:00:00
Training end: 2023-12-31 23:00:00
Possible training hours: 8,584
Active training hours: 8,580
Hours without records: 4


,event_count
0.000,1.000
0.250,5.000
0.500,7.000
0.750,12.000
0.900,22.000
0.950,40.000
0.990,159.210
0.995,427.945
1.000,62872.000


### Burst-threshold decision

The 99th percentile of active training-hour volume is 159.21 events. We round upward and define:

```text
burst threshold = 160 events per active hour

In [22]:
importlib.reload(aggregation)

synthetic_hourly_regime_input = pd.DataFrame({
    "hour_start": pd.to_datetime([
        "2023-01-01 00:00:00",
        "2023-01-01 01:00:00",
        "2023-01-01 02:00:00",
        "2023-01-01 04:00:00",
        "2023-01-01 05:00:00",
        "2023-01-01 06:00:00",
        "2023-01-01 07:00:00",
    ]),
    "data_split": ["train"] * 7,
    "event_count": [10, 200, 20, 10, 200, 200, 10],
})

synthetic_regimes = (
    aggregation.assign_hourly_volume_regimes(
        hourly_summary=synthetic_hourly_regime_input,
        threshold_event_count=160,
        training_quantile=0.99,
    )
)

assert synthetic_regimes["volume_regime"].tolist() == [
    "ordinary",
    "burst",
    "post_burst",
    "ordinary",
    "burst",
    "burst",
    "post_burst",
]

print("Burst assignment: passed")
print("Post-burst assignment: passed")
print("Missing-hour gap handling: passed")
print("Consecutive-burst priority:: passed")

synthetic_regimes[
    ["hour_start", "event_count", "volume_regime"]
]

Burst assignment: passed
Post-burst assignment: passed
Missing-hour gap handling: passed
Consecutive-burst priority:: passed


,hour_start,event_count,volume_regime
0,2023-01-01 00:00:00,10,ordinary
1,2023-01-01 01:00:00,200,burst
2,2023-01-01 02:00:00,20,post_burst
3,2023-01-01 04:00:00,10,ordinary
4,2023-01-01 05:00:00,200,burst
5,2023-01-01 06:00:00,200,burst
6,2023-01-01 07:00:00,10,post_burst


## 13. Assigning behavioural regimes

The threshold is frozen from training data before it is applied to validation and test hours.

Rules:

1. `burst`: the active hour contains at least 160 events.
2. `post_burst`: the hour is below the threshold and immediately follows a burst hour.
3. `ordinary`: every other active hour.
4. A missing hour breaks the post-burst sequence.
5. A burst always takes priority over post-burst.

This creates a label-independent description of system activity, not a prediction of attack or benign behaviour.

In [26]:
import src.regimes as regimes
importlib.reload(regimes)

<module 'src.regimes' from 'c:\\Users\\diyas\\Desktop\\PDS-Log-IDS-Project\\src\\regimes.py'>

In [27]:
burst_threshold_contract = (
    regimes.calculate_burst_threshold(
        verified_hourly,
        training_quantile=0.99,
    )
)

assert np.isclose(
    burst_threshold_contract["raw_threshold"],
    159.21,
)

assert (
    burst_threshold_contract["threshold_event_count"]
    == 160
)

hourly_regimes = (
    regimes.assign_hourly_volume_regimes(
        hourly_summary=verified_hourly,
        threshold_event_count=burst_threshold_contract[
            "threshold_event_count"
        ],
        training_quantile=burst_threshold_contract[
            "training_quantile"
        ],
    )
)

assert len(hourly_regimes) == len(verified_hourly)
assert int(hourly_regimes["event_count"].sum()) == EXPECTED_ROWS

assert not hourly_regimes.duplicated(
    ["hour_start", "data_split"]
).any()

assert set(hourly_regimes["volume_regime"]) == {
    "ordinary",
    "burst",
    "post_burst",
}

regime_distribution = (
    hourly_regimes
    .groupby(
        ["data_split", "volume_regime"],
        observed=True,
    )
    .agg(
        active_hour_count=("hour_start", "size"),
        event_count=("event_count", "sum"),
    )
    .reset_index()
)

regime_distribution["active_hour_percent"] = (
    100
    * regime_distribution["active_hour_count"]
    / regime_distribution
      .groupby("data_split")["active_hour_count"]
      .transform("sum")
)

regime_distribution["event_percent"] = (
    100
    * regime_distribution["event_count"]
    / regime_distribution
      .groupby("data_split")["event_count"]
      .transform("sum")
)

print("Threshold contract:", burst_threshold_contract)
print("Hourly event reconciliation: passed")
print("Regime-grain uniqueness: passed")

regime_distribution

Threshold contract: {'source_split': 'train', 'training_quantile': 0.99, 'raw_threshold': 159.20999999999913, 'threshold_event_count': 160, 'training_active_hours': 8580}
Hourly event reconciliation: passed
Regime-grain uniqueness: passed


,data_split,volume_regime,active_hour_count,event_count,active_hour_percent,event_percent
0,test,burst,19,1310598,4.185022,99.617370
1,test,ordinary,429,4911,94.493392,0.373281
2,test,post_burst,6,123,1.321586,0.009349
3,train,burst,86,169280,1.002331,63.578865
4,train,ordinary,8421,95646,98.146853,35.923110
5,train,post_burst,73,1326,0.850816,0.498024
6,validation,burst,14,473628,1.881720,98.574542
7,validation,ordinary,723,6725,97.177419,1.399651
8,validation,post_burst,7,124,0.940860,0.025808


## 14. Regime-export contract test

A reliable pipeline must test expected failures as well as successful execution.

This test deliberately supplies an incorrect source fingerprint. The exporter must reject it before producing an artifact. It then uses the correct fingerprint and verifies atomic creation, source-column preservation, and idempotent reuse.

In [28]:
import src.regimes as regimes

importlib.reload(regimes)

with tempfile.TemporaryDirectory(
    prefix="_regime_export_test_",
    dir=AGGREGATED_DIRECTORY,
) as temporary_directory:
    test_root = Path(temporary_directory)
    test_hourly_file = test_root / "hourly.csv"
    test_aggregation_manifest = (
        test_root / "aggregation_manifest.json"
    )
    test_output_directory = test_root / "outputs"

    synthetic_hourly_regime_input.to_csv(
        test_hourly_file,
        index=False,
        date_format="%Y-%m-%d %H:%M:%S",
    )

    actual_source_sha256 = (
        aggregation.calculate_file_sha256(
            test_hourly_file
        )
    )

    incorrect_manifest = {
        "outputs": {
            "hourly_summary": {
                "sha256": "0" * 64
            }
        }
    }

    test_aggregation_manifest.write_text(
        json.dumps(incorrect_manifest, indent=2),
        encoding="utf-8",
    )

    try:
        regimes.export_hourly_regimes(
            project_root=PROJECT_ROOT,
            hourly_summary_file=test_hourly_file,
            aggregation_manifest_file=(
                test_aggregation_manifest
            ),
            output_directory=test_output_directory,
            expected_events=650,
            training_quantile=0.99,
        )
    except ValueError as error:
        assert "fingerprint" in str(error)
    else:
        raise AssertionError(
            "Incorrect source fingerprint was accepted."
        )

    correct_manifest = {
        "outputs": {
            "hourly_summary": {
                "sha256": actual_source_sha256
            }
        }
    }

    test_aggregation_manifest.write_text(
        json.dumps(correct_manifest, indent=2),
        encoding="utf-8",
    )

    first_regime_export = regimes.export_hourly_regimes(
        project_root=PROJECT_ROOT,
        hourly_summary_file=test_hourly_file,
        aggregation_manifest_file=(
            test_aggregation_manifest
        ),
        output_directory=test_output_directory,
        expected_events=650,
        training_quantile=0.99,
    )

    second_regime_export = regimes.export_hourly_regimes(
        project_root=PROJECT_ROOT,
        hourly_summary_file=test_hourly_file,
        aggregation_manifest_file=(
            test_aggregation_manifest
        ),
        output_directory=test_output_directory,
        expected_events=650,
        training_quantile=0.99,
    )

    exported_regimes = pd.read_csv(
        test_output_directory / "cj_hourly_regimes.csv",
        parse_dates=["hour_start"],
    )

    pd.testing.assert_frame_equal(
        synthetic_hourly_regime_input.reset_index(drop=True),
        exported_regimes[
            synthetic_hourly_regime_input.columns
        ].reset_index(drop=True),
    )

    assert first_regime_export["export_status"] == "created"
    assert second_regime_export["export_status"] == "reused"
    assert first_regime_export[
        "threshold_uses_weak_labels"
    ] is False
    assert first_regime_export["event_count"] == 650

print("Incorrect-source rejection: passed")
print("Atomic regime export: passed")
print("Source-column preservation: passed")
print("Verified-output reuse: passed")
print("Weak-label independence: passed")
print("Synthetic temporary files removed: passed")

Incorrect-source rejection: passed
Atomic regime export: passed
Source-column preservation: passed
Verified-output reuse: passed
Weak-label independence: passed
Synthetic temporary files removed: passed


## 15. Exporting the behavioural-regime view

The regime file is a downstream derivative of the verified hourly summary.

Its manifest records:

- the hourly-summary fingerprint;
- the regime code fingerprint;
- the training-only thresholdLaughing rule;
- the number of represented events;
- the regimeavis distribution;
- confirmation that weak labels were not used to choose the threshold.

This gives the behavioural interpretation a separate provenance chain from the base aggregates.

In [29]:
importlib.reload(regimes)

regime_export_result = regimes.export_hourly_regimes(
    project_root=PROJECT_ROOT,
    hourly_summary_file=(
        AGGREGATED_DIRECTORY / "cj_hourly_summary.csv"
    ),
    aggregation_manifest_file=(
        AGGREGATED_DIRECTORY
        / "cj_aggregation_manifest.json"
    ),
    output_directory=AGGREGATED_DIRECTORY,
    expected_events=EXPECTED_ROWS,
    training_quantile=0.99,
)

In [33]:
print(
    "Export status:",
    regime_export_result["export_status"],
)
print(
    "Regime version:",
    regime_export_result["regime_version"],
)
print(
    "Output rows:",
    f"{regime_export_result['output_rows']:,}",
)
print(
    "Represented events:",
    f"{regime_export_result['event_count']:,}",
)
print(
    "Threshold contract:",
    regime_export_result["threshold_contract"],
)
print(
    "Weak labels used for threshold:",
    regime_export_result[
        "threshold_uses_weak_labels"
    ],
)
print(
    "Output SHA-256:",
    regime_export_result["output_sha256"],
)

pd.DataFrame(
    regime_export_result["regime_distribution"]
)


Export status: created
Regime version: active-hour-quantile-v1
Output rows: 9,778
Represented events: 2,062,361
Threshold contract: {'source_split': 'train', 'training_quantile': 0.99, 'raw_threshold': 159.20999999999913, 'threshold_event_count': 160, 'training_active_hours': 8580}
Weak labels used for threshold: False
Output SHA-256: e039387cb879f756ae2cb521d770204341f95a8dd6f899813b19df0d87b29e8b


,data_split,volume_regime,active_hour_count,event_count,active_hour_percent,event_percent
0,test,burst,19,1310598,4.185022,99.617370
1,test,ordinary,429,4911,94.493392,0.373281
2,test,post_burst,6,123,1.321586,0.009349
3,train,burst,86,169280,1.002331,63.578865
4,train,ordinary,8421,95646,98.146853,35.923110
5,train,post_burst,73,1326,0.850816,0.498024
6,validation,burst,14,473628,1.881720,98.574542
7,validation,ordinary,723,6725,97.177419,1.399651
8,validation,post_burst,7,124,0.940860,0.025808


## 16. Independent regime verification

The saved regime artifact is reconstructed from the verified hourly summary and compared row by row.

This proves that:

- the training-only threshold can be reproduced;
- every base hourly field remains unchanged;
- only regime-description columns were added;
- all 2,062,361 events remain represented;
- the source, output, and code fingerprints match;
- no partial artifact remains.

In [35]:
HOURLY_REGIME_FILE = (
    AGGREGATED_DIRECTORY / "cj_hourly_regimes.csv"
)

HOURLY_REGIME_MANIFEST = (
    AGGREGATED_DIRECTORY
    / "cj_hourly_regimes_manifest.json"
)

with HOURLY_REGIME_MANIFEST.open(
    encoding="utf-8"
) as file:
    saved_regime_manifest = json.load(file)

saved_hourly_regimes = pd.read_csv(
    HOURLY_REGIME_FILE,
    parse_dates=["hour_start"],
)

saved_hourly_source = pd.read_csv(
    AGGREGATED_DIRECTORY / "cj_hourly_summary.csv",
    parse_dates=["hour_start"],
)

assert (
    aggregation.calculate_file_sha256(
        HOURLY_REGIME_FILE
    )
    == saved_regime_manifest["output_sha256"]
)

assert (
    aggregation.calculate_file_sha256(
        AGGREGATED_DIRECTORY
        / "cj_hourly_summary.csv"
    )
    == saved_regime_manifest["source_sha256"]
)

assert (
    aggregation.calculate_file_sha256(
        PROJECT_ROOT / "src" / "regimes.py"
    )
    == saved_regime_manifest["regime_code_sha256"]
)

assert (
    aggregation.calculate_file_sha256(
        PROJECT_ROOT / "src" / "aggregation.py"
    )
    == saved_aggregation_manifest[
        "aggregation_code_sha256"
    ]
)

reconstructed_threshold = (
    regimes.calculate_burst_threshold(
        saved_hourly_source[
            ["hour_start", "data_split", "event_count"]
        ],
        training_quantile=0.99,
    )
)

assert (
    reconstructed_threshold
    == saved_regime_manifest["threshold_contract"]
)

reconstructed_regimes = (
    regimes.assign_hourly_volume_regimes(
        hourly_summary=saved_hourly_source,
        threshold_event_count=reconstructed_threshold[
            "threshold_event_count"
        ],
        training_quantile=reconstructed_threshold[
            "training_quantile"
        ],
    )
)

pd.testing.assert_frame_equal(
    saved_hourly_regimes,
    reconstructed_regimes,
    check_dtype=False,
)

pd.testing.assert_frame_equal(
    saved_hourly_source,
    saved_hourly_regimes[
        saved_hourly_source.columns
    ],
    check_dtype=False,
)

assert len(saved_hourly_regimes) == 9_778

assert (
    int(saved_hourly_regimes["event_count"].sum())
    == EXPECTED_ROWS
)

assert not saved_hourly_regimes.duplicated(
    ["hour_start", "data_split"]
).any()

assert set(saved_hourly_regimes["volume_regime"]) == {
    "ordinary",
    "burst",
    "post_burst",
}

assert saved_regime_manifest[
    "threshold_uses_weak_labels"
] is False

assert saved_regime_manifest[
    "threshold_contract"
]["source_split"] == "train"

assert saved_regime_manifest[
    "threshold_contract"
]["threshold_event_count"] == 160

partial_files = list(
    AGGREGATED_DIRECTORY.glob("*.partial")
)

assert not partial_files

print("Regime rows:", f"{len(saved_hourly_regimes):,}")
print("Represented events:", f"{EXPECTED_ROWS:,}")
print("Threshold reconstruction: passed")
print("Exact regime reconstruction: passed")
print("Base-hour preservation: passed")
print("Source, output, and code fingerprints: passed")
print("Temporal threshold isolation: passed")
print("Partial files remaining:", len(partial_files))

Regime rows: 9,778
Represented events: 2,062,361
Threshold reconstruction: passed
Exact regime reconstruction: passed
Base-hour preservation: passed
Source, output, and code fingerprints: passed
Temporal threshold isolation: passed
Partial files remaining: 0


## Practical 7 conclusion

Practical 7 transformed 2,062,361 verified event records into multiple analysis-ready views while preserving privacy, provenance, temporal boundaries, and exact event totals.

### Generated views

| View | Rows | Meaning of one row |
|---|---:|---|
| Hourly summary | 9,778 | One active hour within one temporal split |
| Daily summary | 408 | One calendar day within one temporal split |
| Client-split summary | 17,607 | One pseudonymous client within one split |
| Client-day summary | 32,809 | One pseudonymous client on one day |
| Hourly regimes | 9,778 | One active hour with an ordinary, burst, or post-burst regime |

### Important result

The training-only 99th-percentile rule produced a burst threshold of 160 events per active hour.

- Training bursts: 1.00% of active hours but 63.58% of events.
- Validation bursts: 1.88% of active hours but 98.57% of events.
- Test bursts: 4.19% of active hours but 99.62% of events.

This demonstrates strong temporal concentration and distribution drift. It does not by itself prove that burst hours are attacks.

### Distinctive engineering decisions

- Exact alignment using `event_id` and `record_hash`.
- Chunk-invariant aggregation across large files.
- Keyed HMAC client pseudonymisation.
- Separate event-weighted and client-centred views.
- Training-only threshold selection.
- Weak-label-independent regime construction.
- Atomic outputs with independently verified manifests.

### Limitations

- Weak labels are not authoritative ground truth.
- The volume regime is global and may be dominated by high-volume clients.
- Inactive hours are excluded from the active-hour threshold.
- Aggregates are descriptive and are not automatically safe as model features.
- Client pseudonyms remain linkable when the same private HMAC key is used.

Practical 8 will use these verified views for EDA and visualisation without changing the underlying data.